In [15]:
import pandas as pd
import numpy as np
import psutil
from unidecode import unidecode
from fuzzywuzzy import fuzz
from collections import defaultdict
import json

# Check memory usage
def memory_usage():
    process = psutil.Process()
    mem_info = process.memory_info()
    return mem_info.rss / (1024 * 1024)  # Convert bytes to MB

print(f"Memory usage: {memory_usage()} MB")

Memory usage: 164.578125 MB


In [2]:
# Set pandas display options
pd.set_option('display.max_rows', None)  # Show all rows
pd.set_option('display.max_columns', None)  # Show all columns
pd.set_option('display.max_colwidth', None)  # Show full column width
pd.set_option('display.expand_frame_repr', False)  # Disable wrapping of rows

In [ ]:
main = pd.read_csv('./classifier_positions.csv', encoding='utf-8', sep=',', header=None)
main.columns = ["1", "2", "3", "4", "5", "6", "7", "8", "9", "10", "11", "12", "13", "14", "15"]

,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15
0,1,NaN,1.,1,C8,Авербандщик,Авербандчи,8264,П,2 — 5,ССПО,3320900,Averbandchi,i,1
1,2,NaN,2.,2,C8,Авиационный механик (техник) по обслуживанию и ремонту воздушных судов.,Ҳаво кемаларига хизмат кўрсатиш ва уларни таъмирлаш бўйича авиация механиги (техник),7232,П,2 — 6,ССПО,3310400,havo kemalariga xizmat ko‘rsatish va ularni ta’mirlash bo‘yicha aviatsiya mexanigi (texnik),i,2
2,3,NaN,3.,3,C8,Авиационный механик перронно-технической бригады (ПТБ) аэропорта,Аэропорт перрон-техника бригадаси авиация механиги,7232,П,2 — 6,ССПО,3310400,Aeroport perron-texnika brigadasi aviatsiya mexanigi,i,3
3,4,NaN,4.,4,C8,Авиационный механик (техник) по планеру и двигателям,Планер ва двигателлар бўйича авиация механиги (техник),7232,П,2 — 6,ССПО,3310400,Planer va dvigatellar bo‘yicha aviatsiya mexanigi (texnik),i,4
4,5,NaN,5.,5,C8,Авиационный механик (техник) по приборам и электрооборудованию,Приборлар ва электр жихозлар бўйича авиация механиги (техник),7232,П,2 — 6,ССПО,3310400,Priborlar va elektr jixoz lar bo‘yicha aviatsiya mexanigi (texnik),i,5


In [7]:
main.head(50)

,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15
0,1,NaN,1.,1,C8,Авербандщик,Авербандчи,8264,П,2 — 5,ССПО,3320900,Averbandchi,i,1
1,2,NaN,2.,2,C8,Авиационный механик (техник) по обслуживанию и ремонту воздушных судов.,Ҳаво кемаларига хизмат кўрсатиш ва уларни таъмирлаш бўйича авиация механиги (техник),7232,П,2 — 6,ССПО,3310400,havo kemalariga xizmat ko‘rsatish va ularni ta’mirlash bo‘yicha aviatsiya mexanigi (texnik),i,2
2,3,NaN,3.,3,C8,Авиационный механик перронно-технической бригады (ПТБ) аэропорта,Аэропорт перрон-техника бригадаси авиация механиги,7232,П,2 — 6,ССПО,3310400,Aeroport perron-texnika brigadasi aviatsiya mexanigi,i,3
3,4,NaN,4.,4,C8,Авиационный механик (техник) по планеру и двигателям,Планер ва двигателлар бўйича авиация механиги (техник),7232,П,2 — 6,ССПО,3310400,Planer va dvigatellar bo‘yicha aviatsiya mexanigi (texnik),i,4
4,5,NaN,5.,5,C8,Авиационный механик (техник) по приборам и электрооборудованию,Приборлар ва электр жихозлар бўйича авиация механиги (техник),7232,П,2 — 6,ССПО,3310400,Priborlar va elektr jixoz lar bo‘yicha aviatsiya mexanigi (texnik),i,5
5,6,NaN,6.,6,C8,Авиационный механик (техник) по радиооборудованию,Радио жихозлар бўйича авиация механиги (техник),7232,П,2 — 6,ССПО,3310400,Radio jixoz lar bo‘yicha aviatsiya mexanigi (texnik),i,6
6,7,NaN,7.,7,C8,Авиационный техник (механик) по парашютным и аварийно-спасательным средствам,Парашют ва авария-қутқарув воситалари бўйича авиация техниги (механик),7232,П,4 — 6,ССПО,3310400,Parashyut va avariya-qutqaruv vositalari bo‘yicha aviatsiya texnigi (mexanik),i,7
7,8,NaN,8.,8,C8,Авиационный техник по горюче-смазочным материалам,Ёқилғи-мойлаш материаллари бўйича авиация техниги,7232,П,3 — 5,ССПО,3310400,Yoqilg‘i-moylash materiallari bo‘yicha aviatsiya texnigi,i,8
8,9,NaN,9.,9,C8,Автоклавщик в гидролизном производстве,Гидролиз ишлаб чиқаришда автоклавчи,8152,П,3 — 4,ССПО,3320400,Gidroliz ishlab chiqarishda avtoklavchi,i,9
9,10,NaN,10.,10,C8,Автоклавщик в производстве стекла и стеклоизделий,Шиша ва шиша буюмлар ишлаб чиқаришда автоклавчи,8132,П,3 — 4,ССПО,"3340500, 3340200, 3320400",Shisha va shisha buyumlar ishlab chiqarishda avtoklavchi,i,10


In [12]:
poss = ["Охранник", 
		"Дежурный", 
		"Старший смены охраны", 
		"Начальник охраны", 
		"Начальник службы охраны", 
		"Начальник охраны объекта", 
		"Начальник охраны (военизированной, пожарной, сторожевой)", 
		"Хранитель экспонатов", "Сторож",
		"Дежурный автостанции",
		"Дежурный администратор",
		"Дежурный аэропорта",
		"Дежурный бюро пропусков",
		"Дежурный гостиницы",
		"Дежурный диспетчер",
		"Дежурный инкассатор",
		"Дежурный камеры хранения",
		"Дежурный по вокзалу",
		"Дежурный по направлению",
		"Дежурный по общежитию",
		"Дежурный по полетам",
		"Дежурный по путям",
		"Дежурный по разъезду",
		"Дежурный по режиму",
		"Дежурный по станции",
		"Дежурный по электродепо",
		"Дежурный по электросвязи",
		"Дежурный службы движения",
		"Инспектор военизированной охраны",
		"Инспектор по охране природы",
		"Старший смены охраны"]
fltr = main[main["6"].isin(poss)]

In [16]:
fltr.to_excel('security_officer_names.xlsx', index=False, encoding='utf-8-sig')

/Users/ilkhom/miniconda3/lib/python3.9/site-packages/pandas/util/_decorators.py:211: FutureWarning: the 'encoding' keyword is deprecated and will be removed in a future version. Please take steps to stop the use of 'encoding'
  return func(*args, **kwargs)
